# GeoGram: Integrace dat z RÚIAN a ČSÚ

Tento notebook demonstruje, jak se napojují:
- UI_OBEC.csv (seznam obcí)
- UI_OKRES.csv (správní okresy)
- UI_VUSC.csv (kraje)
- Tabulka populací od ČSÚ

Výsledek: kompletní tabulka všech českých obcí s demografi. 📊

In [1]:
import sys
from pathlib import Path

# Přidat root projektu na path
project_root = Path('.').resolve().parent
sys.path.insert(0, str(project_root / 'src'))

from geogram.data import load_all_municipalities, filter_suffix_ice
import pandas as pd

# Načíst kompletní data (s správnou cestou k datům)
data_dir = project_root / 'data' / 'raw'
df_all = load_all_municipalities(data_dir=data_dir)
print(f'Celkem obcí: {len(df_all):,}')
print(f'Sloupce: {list(df_all.columns)}')
print(f'\nPrvních 10 řádků:')
df_all.head(10)

Celkem obcí: 6,259
Sloupce: ['code', 'name', 'district_code', 'district_name', 'region_code', 'region_name', 'status_code', 'population_total', 'population_men', 'population_women', 'avg_age_total']

Prvních 10 řádků:


,code,name,district_code,district_name,region_code,region_name,status_code,population_total,population_men,population_women,avg_age_total
0,554782,Praha,9999,území Hlavního města Prahy,19,Hlavní město Praha,5,1397880.0,679162.0,718718.0,41.864895
1,529303,Benešov,3201,Benešov,27,Středočeský kraj,3,17043.0,8058.0,8985.0,43.878631
2,532568,Bernartice,3201,Benešov,27,Středočeský kraj,2,223.0,102.0,121.0,45.172646
3,532380,Blažejovice,3201,Benešov,27,Středočeský kraj,2,102.0,50.0,52.0,49.843137
4,532096,Borovnice,3201,Benešov,27,Středočeský kraj,2,91.0,42.0,49.0,41.247253
5,532924,Bukovany,3201,Benešov,27,Středočeský kraj,2,824.0,429.0,395.0,41.003641
6,529451,Bystřice,3201,Benešov,27,Středočeský kraj,3,4698.0,2361.0,2337.0,42.071307
7,530743,Bílkovice,3201,Benešov,27,Středočeský kraj,2,225.0,113.0,112.0,45.780000
8,532878,Chleby,3201,Benešov,27,Středočeský kraj,2,49.0,28.0,21.0,51.459184
9,529770,Chlum,3201,Benešov,27,Středočeský kraj,2,123.0,63.0,60.0,46.930894


In [2]:
# Filtrovat obce na -ice
df_ice = filter_suffix_ice(df_all, name_column='name')
print(f'Obcí končících na -ice: {len(df_ice):,}')
print(f'Podíl z celku: {len(df_ice)/len(df_all)*100:.1f}%')

# Seřadit podle populace
df_ice_sorted = df_ice.sort_values('population_total', ascending=False, na_position='last')
print(f'\nTop 20 obcí na -ice (podle populace):')
df_ice_sorted[['name', 'district_name', 'region_name', 'population_total', 'avg_age_total']].head(20)

Obcí končících na -ice: 1,806
Podíl z celku: 28.9%

Top 20 obcí na -ice (podle populace):


,name,district_name,region_name,population_total,avg_age_total
1247,České Budějovice,České Budějovice,Jihočeský kraj,97231.0,43.348824
3584,Pardubice,Pardubice,Pardubický kraj,92319.0,43.442742
2731,Teplice,Teplice,Ústecký kraj,50912.0,43.559082
2546,Litoměřice,Litoměřice,Ústecký kraj,22767.0,43.822221
1631,Strakonice,Strakonice,Jihočeský kraj,22355.0,44.568888
5827,Kopřivnice,Nový Jičín,Moravskoslezský kraj,21374.0,43.822541
5469,Hranice,Přerov,Olomoucký kraj,17969.0,44.129306
6002,Otrokovice,Zlín,Zlínský kraj,17401.0,45.693897
517,Neratovice,Mělník,Středočeský kraj,16360.0,43.718337
706,Milovice,Nymburk,Středočeský kraj,14270.0,35.642957


In [3]:
# Statistika
print('Statistika -ice obcí:')
print(f'  Součet populace: {df_ice["population_total"].sum():,.0f}')
print(f'  Průměr: {df_ice["population_total"].mean():.0f}')
print(f'  Medián: {df_ice["population_total"].median():.0f}')
print(f'  Min: {df_ice["population_total"].min():.0f}')
print(f'  Max: {df_ice["population_total"].max():.0f}')

# Rozložení po krajích
print(f'\nRozdělení podle krajů:')
print(df_ice.groupby('region_name').size().sort_values(ascending=False))

Statistika -ice obcí:
  Součet populace: 1,923,253
  Průměr: 1065
  Medián: 463
  Min: 0
  Max: 97231

Rozdělení podle krajů:
region_name
Středočeský kraj        337
Jihomoravský kraj       276
Jihočeský kraj          192
Kraj Vysočina           158
Královéhradecký kraj    131
Plzeňský kraj           120
Olomoucký kraj          113
Pardubický kraj         112
Ústecký kraj            107
Moravskoslezský kraj    100
Zlínský kraj             99
Liberecký kraj           42
Karlovarský kraj         19
dtype: int64


In [ ]:
# Uložit do processed
output_path = project_root / 'data' / 'processed' / 'municipalities_ice_integrated.csv'
output_path.parent.mkdir(parents=True, exist_ok=True)
df_ice_sorted.to_csv(output_path, index=False, encoding='utf-8')
print(f'Uloženo: {output_path}')

Uloženo: C:\Users\dobes\Documents\UniversityCodingProject\NewFunnyProjects\GeoGram_sufix-ice\data\processed\municipalities_ice_integrated.csv


: 